<a href="https://colab.research.google.com/github/PratikBiswas99/End-to-End-ML-Model/blob/main/Welcome_To_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**ML Model**

In [2]:
import kagglehub
path = kagglehub.dataset_download("datasnaek/youtube-new")

Using Colab cache for faster access to the 'youtube-new' dataset.


In [3]:
import os

print("Files in dataset:", os.listdir(path))

Files in dataset: ['GBvideos.csv', 'MXvideos.csv', 'KRvideos.csv', 'MX_category_id.json', 'GB_category_id.json', 'US_category_id.json', 'IN_category_id.json', 'DEvideos.csv', 'KR_category_id.json', 'RU_category_id.json', 'FRvideos.csv', 'USvideos.csv', 'INvideos.csv', 'RUvideos.csv', 'CA_category_id.json', 'DE_category_id.json', 'JP_category_id.json', 'JPvideos.csv', 'FR_category_id.json', 'CAvideos.csv']


In [4]:
import os
import pandas as pd

# Load the Indian trending dataset as an example
df_in = pd.read_csv(os.path.join(path, "INvideos.csv"))

# Inspect columns, non-null counts, and data types
df_in.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 37352 entries, 0 to 37351
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   video_id                37352 non-null  object
 1   trending_date           37352 non-null  object
 2   title                   37352 non-null  object
 3   channel_title           37352 non-null  object
 4   category_id             37352 non-null  int64 
 5   publish_time            37352 non-null  object
 6   tags                    37352 non-null  object
 7   views                   37352 non-null  int64 
 8   likes                   37352 non-null  int64 
 9   dislikes                37352 non-null  int64 
 10  comment_count           37352 non-null  int64 
 11  thumbnail_link          37352 non-null  object
 12  comments_disabled       37352 non-null  bool  
 13  ratings_disabled        37352 non-null  bool  
 14  video_error_or_removed  37352 non-null  bool  
 15  de

In [5]:
# Check exact missing counts and percentages
missing_df = pd.DataFrame({
    'Missing Count': df_in.isnull().sum(),
    'Missing Percentage': (df_in.isnull().sum() / len(df_in)) * 100
})
print(missing_df[missing_df['Missing Count'] > 0])

             Missing Count  Missing Percentage
description            561            1.501928


In [6]:
# Fill missing descriptions with a default placeholder string
df_in['description'] = df_in['description'].fillna('No description provided')

# Verify that 0 nulls remain in the dataframe
print("Total missing values remaining:", df_in.isnull().sum().sum())

Total missing values remaining: 0


In [9]:
# Check numerical distribution summary
print(df_in[['views', 'likes', 'dislikes', 'comment_count']].describe())

              views         likes      dislikes  comment_count
count  3.735200e+04  3.735200e+04  3.735200e+04    37352.00000
mean   1.060478e+06  2.708272e+04  1.665082e+03     2676.99743
std    3.184932e+06  9.714510e+04  1.607617e+04    14868.31713
min    4.024000e+03  0.000000e+00  0.000000e+00        0.00000
25%    1.239155e+05  8.640000e+02  1.080000e+02       81.00000
50%    3.045860e+05  3.069000e+03  3.260000e+02      329.00000
75%    7.992912e+05  1.377425e+04  1.019250e+03     1285.00000
max    1.254322e+08  2.912710e+06  1.545017e+06   827755.00000


In [10]:
# Calculate Q1, Q3, and IQR for views
Q1 = df_in['views'].quantile(0.25)
Q3 = df_in['views'].quantile(0.75)
IQR = Q3 - Q1

# Define bounds (using 3.0 to safely retain legitimate viral spikes)
lower_bound = Q1 - (3.0 * IQR)
upper_bound = Q3 + (3.0 * IQR)

# Filter or cap the dataset
# Option A: Filter them out (be careful not to lose genuine viral trends)
df_clean = df_in[(df_in['views'] >= lower_bound) & (df_in['views'] <= upper_bound)]

print(f"Original rows: {len(df_in)}, Rows after outlier filter: {len(df_clean)}")

Original rows: 37352, Rows after outlier filter: 34643


In [11]:
# Count exact full-row duplicates
print("Exact duplicate rows:", df_clean.duplicated().sum())

Exact duplicate rows: 3775


In [12]:
# Drop exact full-row duplicates from df_clean
df_clean = df_clean.drop_duplicates()

print(f"Final rows after cleaning missing data, outliers, and duplicates: {len(df_clean)}")

Final rows after cleaning missing data, outliers, and duplicates: 30868


In [17]:
# Convert trending_date and publish_time to actual datetime objects
# Note: YouTube trending dates in this dataset are usually formatted as YY.DD.MM
df_clean['trending_date'] = pd.to_datetime(df_clean['trending_date'], format='%y.%d.%m')
df_clean['publish_time'] = pd.to_datetime(df_clean['publish_time'])

# Verify the type conversion
print(df_clean[['trending_date', 'publish_time']].dtypes)

trending_date         datetime64[ns]
publish_time     datetime64[ns, UTC]
dtype: object


In [18]:
df_clean['title'] = df_clean['title'].str.strip()
df_clean['channel_title'] = df_clean['channel_title'].str.strip()

In [19]:
def validate_youtube_data(df):
    # Rule 1: Check for negative values in metrics
    assert (df['views'] >= 0).all(), "Validation Error: Negative views detected!"
    assert (df['likes'] >= 0).all(), "Validation Error: Negative likes detected!"
    assert (df['dislikes'] >= 0).all(), "Validation Error: Negative dislikes detected!"
    assert (df['comment_count'] >= 0).all(), "Validation Error: Negative comment counts detected!"

    # Rule 2: Check for nulls in critical identifier columns
    critical_cols = ['video_id', 'title', 'channel_title', 'trending_date']
    for col in critical_cols:
        assert df[col].isnull().sum() == 0, f"Validation Error: Null values found in critical column {col}!"

    # Rule 3: Verify row count integrity
    assert len(df) > 0, "Validation Error: Dataset is empty!"

    print("All data validation checks passed successfully!")

# Run the validation on your cleaned dataframe
validate_youtube_data(df_clean)

All data validation checks passed successfully!


In [20]:
# Compute correlation matrix for numerical engagement columns
correlation_matrix = df_clean[['views', 'likes', 'dislikes', 'comment_count']].corr()
print("Correlation Matrix:")
print(correlation_matrix)

Correlation Matrix:
                  views     likes  dislikes  comment_count
views          1.000000  0.575153  0.566488       0.201081
likes          0.575153  1.000000  0.582543       0.523886
dislikes       0.566488  0.582543  1.000000       0.266341
comment_count  0.201081  0.523886  0.266341       1.000000


In [21]:
# Count how many trending videos appear per category_id
top_categories = df_clean['category_id'].value_counts().head(10)
print("Top 10 Trending Category IDs:")
print(top_categories)

Top 10 Trending Category IDs:
category_id
24    13993
25     4669
23     2806
10     2545
22     2337
1      1194
27     1165
26      741
17      540
28      505
Name: count, dtype: int64


In [22]:
top_channels = df_clean['channel_title'].value_counts().head(10)
print("Top 10 Channels by Trending Frequency:")
print(top_channels)

Top 10 Channels by Trending Frequency:
channel_title
VikatanTV             208
SAB TV                206
ETV Plus India        206
Study IQ education    202
Flowers Comedy        202
etvteluguindia        201
Tarang TV             199
Mazhavil Manorama     196
RadaanMedia           193
V6 News Telugu        190
Name: count, dtype: int64


In [24]:
# Calculate how many days it took from publishing to trending
df_clean['days_to_trend'] = (df_clean['trending_date'] - df_clean['publish_time'].dt.tz_localize(None)).dt.days

# Extract publish day of the week (0 = Monday, 6 = Sunday) and publish hour
df_clean['publish_dow'] = df_clean['publish_time'].dt.dayofweek
df_clean['publish_hour'] = df_clean['publish_time'].dt.hour

In [25]:
# Avoid division by zero by adding a small epsilon or handling zero views
df_clean['like_view_ratio'] = df_clean['likes'] / (df_clean['views'] + 1)
df_clean['comment_view_ratio'] = df_clean['comment_count'] / (df_clean['views'] + 1)
df_clean['dislike_like_ratio'] = df_clean['dislikes'] / (df_clean['likes'] + 1)

In [26]:
# Length of the title and description
df_clean['title_length'] = df_clean['title'].str.len()
df_clean['description_length'] = df_clean['description'].str.len()

# Count number of tags
df_clean['tag_count'] = df_clean['tags'].apply(lambda x: len(str(x).split('|')))

In [27]:
from sklearn.model_selection import train_test_split

# Define your feature set (dropping non-numeric, target, or ID columns)
features = [
    'category_id', 'days_to_trend', 'publish_dow', 'publish_hour',
    'like_view_ratio', 'comment_view_ratio', 'title_length',
    'description_length', 'tag_count'
]
target = 'views'

X = df_clean[features]
y = df_clean[target]

In [28]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

Training set shape: (24694, 9)
Testing set shape: (6174, 9)


In [30]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Now rerun your error calculation
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))

print(f"XGBoost Mean Absolute Error (MAE): {mae_xgb:.2f}")
print(f"XGBoost Root Mean Squared Error (RMSE): {rmse_xgb:.2f}")

XGBoost Mean Absolute Error (MAE): 260896.72
XGBoost Root Mean Squared Error (RMSE): 391996.50


In [31]:
import xgboost as xgb

# 1. Initialize the XGBoost Regressor
# n_estimators sets the number of boosting rounds, learning_rate controls step size
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1
)

# 2. Train the model on your training split
print("Training XGBoost model...")
xgb_model.fit(X_train, y_train)
print("Training complete!")

# 3. Generate predictions on your test split
y_pred_xgb = xgb_model.predict(X_test)

# 4. Calculate error metrics
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))

print(f"XGBoost Mean Absolute Error (MAE): {mae_xgb:.2f}")
print(f"XGBoost Root Mean Squared Error (RMSE): {rmse_xgb:.2f}")

Training XGBoost model...
Training complete!
XGBoost Mean Absolute Error (MAE): 260896.72
XGBoost Root Mean Squared Error (RMSE): 391996.50


In [32]:
import pandas as pd
import matplotlib.pyplot as plt

# Extract and sort feature importances
importances = xgb_model.feature_importances_
feature_imp_df = pd.DataFrame({
    'Feature': features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("Feature Importances:")
print(feature_imp_df)

Feature Importances:
              Feature  Importance
1       days_to_trend    0.436255
0         category_id    0.109038
6        title_length    0.082136
7  description_length    0.077597
3        publish_hour    0.073817
8           tag_count    0.072662
4     like_view_ratio    0.062464
5  comment_view_ratio    0.054230
2         publish_dow    0.031800


In [ ]:
# Try tuning learning_rate, max_depth, and n_estimators
tuned_xgb = xgb.XGBRegressor(
    n_estimators=200,      # More trees
    learning_rate=0.05,    # Slower, more precise learning rate
    max_depth=6,           # Controls tree complexity
    random_state=42,
    n_jobs=-1
)

tuned_xgb.fit(X_train, y_train)
y_pred_tuned = tuned_xgb.predict(X_test)

print(f"Tuned MAE: {mean_absolute_error(y_test, y_pred_tuned):.2f}")
print(f"Tuned RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_tuned)):.2f}")

In [36]:
import joblib

# Save the model to disk
joblib.dump(tuned_xgb, 'youtube_views_xgb_model.pkl')
print("Model successfully saved to disk!")

# (Optional) To load it back later:
# loaded_model = joblib.load('youtube_views_xgb_model.pkl')

Model successfully saved to disk!


In [35]:
import xgboost as xgb
import joblib
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# 1. Train the tuned XGBoost model
tuned_xgb = xgb.XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)

print("Training tuned XGBoost model...")
tuned_xgb.fit(X_train, y_train)
print("Training complete!")

# 2. Predict and evaluate your errors
y_pred_tuned = tuned_xgb.predict(X_test)
print(f"Tuned MAE: {mean_absolute_error(y_test, y_pred_tuned):.2f}")
print(f"Tuned RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_tuned)):.2f}")

# 3. Save the model to disk safely
joblib.dump(tuned_xgb, 'youtube_views_xgb_model.pkl')
print("Model successfully saved to disk!")

Training tuned XGBoost model...
Training complete!
Tuned MAE: 261810.86
Tuned RMSE: 394151.99
Model successfully saved to disk!
